In [1]:
import pandas as pd

print("Membaca dan menggabungkan dataset...")

df1 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_bigmo_27_2_2026_cleaned_labeled.csv')
df2 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_windah_2_3_2026_cleaned_labeled.csv')
df3 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_KPU_Jabar_debat_pilgub_11_11_2024_cleaned_labeled.csv')
df4 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_kobo_1_4_2022_cleaned_labeled.csv')
df5 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_jebret_media_29_4_2024_cleaned_labeled.csv')
df6 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_marsha_19_3_2023_cleaned_labeled.csv')
df7 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_tribun_jogja_sidang_isbat_28_2_2025_cleaned_labeled.csv')
df8 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_dss_music_11_5_2025_cleaned_labeled.csv')
df9 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_dss_music_17_9_2021_cleaned_labeled.csv')
df10 = pd.read_csv('../generate_label/output_from_llm/data_chat_youtube_gadgetin_17_4_2024_cleaned_labeled.csv')
df11 = pd.read_csv('../generate_label/output_from_llm/data_judi_online_komentar_youtube_cleaned_labeled.csv')
df12 = pd.read_csv('../generate_label/output_from_llm/data_judi_online_youtube_wayang_cleaned_labeled.csv')

# Gabungkan dataframe menjadi satu
df_gabungan = pd.concat([df1, df2, df3, df4, df5, df6, df7, df8, df9, df10, df11, df12], ignore_index=True)

df_gabungan.to_csv('../resources/data_gabungan.csv', index=False)
print(f"Total data gabungan mentah: {len(df_gabungan)} baris\n")

Membaca dan menggabungkan dataset...
Total data gabungan mentah: 80437 baris



In [ ]:
import pandas as pd
from collections import Counter

# Load Dataset 
df = pd.read_csv('../resources/data_gabungan.csv')

kolom_teks = 'Pesan_Clean'
df[kolom_teks] = df[kolom_teks].fillna('')

# Ekstrak dan hitung kata
print("Memulai perhitungan kata...")

# Menggabungkan seluruh baris chat menjadi satu teks super panjang, lalu dipisah per kata (split)
semua_kata = ' '.join(df[kolom_teks].tolist()).split()

# Menghitung frekuensi kata
frekuensi_kata = Counter(semua_kata)

print("\n--- 20 Kata Paling Sering Muncul ---")
for kata, jumlah in frekuensi_kata.most_common(20):
    print(f"{kata} : {jumlah} kali")

# Ekspor menjadi kamus SymSpell
nama_file_output = '../resources/count_words_from_dataset.txt'

with open(nama_file_output, 'w', encoding='utf-8') as f:
    # Ambil semua kata yang muncul minimal 10 kali
    for kata, jumlah in frekuensi_kata.items():
        if jumlah >= 10: 
            # Booster multiplier
            jumlah_booster = jumlah * 10
            
            # format: [Kata] [Tab] [Frekuensi]
            f.write(f"{kata}\t{jumlah_booster}\n")

print(f"\nBerhasil! File '{nama_file_output}' telah disimpan.")

Memulai perhitungan kata...

--- 20 Kata Paling Sering Muncul ---
bang : 7535 kali
di : 4072 kali
nya : 2426 kali
ini : 2308 kali
aja : 2023 kali
yg : 1827 kali
no : 1812 kali
ada : 1775 kali
ga : 1771 kali
bg : 1745 kali
3 : 1695 kali
yang : 1632 kali
udah : 1604 kali
gak : 1493 kali
4 : 1386 kali
indonesia : 1380 kali
, : 1363 kali
1 : 1357 kali
itu : 1303 kali
aku : 1284 kali

Berhasil! File '../resources/count_words_from_dataset.txt' telah disimpan.


In [ ]:
# Kamus Hibrida dengan Cut-Off

kamus_hybrid = {}

print("Membaca Korpus Web (Kamus Baku)...")
with open('../resources/ind-id_web_2017_100K-words.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            kata = parts[1].lower()
            try:
                frekuensi = int(parts[2])
                
                if kata.isalpha() and frekuensi >= 20:
                    kamus_hybrid[kata] = frekuensi
            except ValueError:
                continue

print("Membaca Korpus Dataset Anda (Kamus Gaul)...")
with open('../resources/count_words_from_dataset.txt', 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) == 2:
            kata = parts[0].lower()
            try:
                frekuensi_dataset = int(parts[1])
                
                # Minimal 15 kali (150)
                if kata.isalpha() and int(parts[1]) >= 150:
                    if kata in kamus_hybrid:
                        kamus_hybrid[kata] = max(kamus_hybrid[kata], frekuensi_dataset)
                    else:
                        kamus_hybrid[kata] = frekuensi_dataset
            except ValueError:
                continue

print("Menyimpan menjadi Kamus Final...")
with open('../resources/kamus_symspell_final.txt', 'w', encoding='utf-8') as f:
    for kata, freq in kamus_hybrid.items():
        f.write(f"{kata}\t{freq}\n")

print(f"Selesai! Kamus Final berhasil dibuat dengan {len(kamus_hybrid)} kata bersih.")

Membaca Korpus Web (Kamus Baku)...
Membaca Korpus Dataset Anda (Kamus Gaul)...
Menyimpan menjadi Kamus Final...
Selesai! Kamus Final berhasil dibuat dengan 7862 kata bersih.
